# Demonstração: Analista e Validador

Este notebook apresenta o protótipo acadêmico de dois agentes coordenados por LangGraph. O **Analista** escolhe uma consulta, solicita a ferramenta de busca e redige; o **Validador** confere a resposta e pode solicitar uma revisão. O orquestrador aplica os limites e decide o encerramento.

O arquivo está **sem execuções e sem resultados previamente preenchidos**. `EXECUTAR_MODELOS = False` é o padrão: abrir ou executar as células assim não chama modelos, não abre o banco e não cria arquivos de resultados. Os resultados aparecem somente após uma execução deliberada.

No modo sintético, apenas a busca usa material fictício: **os dois agentes continuam dependendo de chamadas reais aos modelos no Ollama**. Não há substituição da LLM por respostas prontas neste notebook.


## Fluxo e colaboração

```mermaid
flowchart TD
    E[Pergunta e guardrail de entrada] --> A[Analista: escolher consulta]
    E -->|Fora do escopo| F[Resposta fixa]
    A --> B[Ferramenta: buscar evidências]
    B -->|Fontes disponíveis| R[Analista: rascunho]
    B -->|Sem evidência| L[Resposta limitada]
    R --> G[Guardrail de saída]
    G --> V[Validador: conferir fontes e resposta]
    V -->|Aprovar e controles aprovados| S[Resposta final]
    V -->|Revisar: no máximo uma vez| C[Analista: corrigir com feedback]
    C --> G
    V -->|Insuficiência ou revisão esgotada| L
```

Se o visualizador não renderizar Mermaid: **Analista → busca → rascunho → Validador → feedback → Analista revisa → Validador confere novamente**. A revisão é opcional e limitada a uma. Guardrails, busca e orquestrador são componentes auxiliares; os agentes são Analista e Validador.


In [ ]:
from pathlib import Path
import importlib.util
import json
import shutil
import subprocess
import sys

# Funciona a partir da raiz ou da pasta deste notebook, sem caminho de máquina fixo.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "worker/evidencia/langgraph/orquestrador.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Abra este notebook a partir de uma pasta dentro do repositório lex_clt.ai.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)


## Preparar o ambiente

Use o Python do ambiente do projeto e instale as dependências do protótipo antes de habilitar os modelos. O kernel deste notebook deve apontar para esse mesmo Python.

```bash
python3 -m venv .venv
# macOS/Linux: source .venv/bin/activate
# Windows PowerShell: .venv\Scripts\Activate.ps1
python -m pip install -r docs/legado/requirements/requirements-multiagentes.txt
# Modo real: instalar também as dependências do recuperador existente.
# python -m pip install -r docs/legado/requirements/requirements.txt
```

O arquivo `docs/legado/requirements/requirements-multiagentes.txt` fixa a versão de LangGraph testada pelo protótipo. Para usar a busca real, instale também `docs/legado/requirements/requirements.txt`.

O Ollama precisa estar disponível no endpoint configurado e os modelos escolhidos precisam estar instalados. Pesos, banco e credenciais não acompanham o clone. O cliente deste protótipo não baixa modelos implicitamente.

Para o modo **real**, também são necessários PostgreSQL/pgvector provisionado, coleções de legislação e jurisprudência validadas, cache local BGE-M3/reranker e visões lexicais preparadas. A busca real abre esses recursos somente na primeira consulta e não cria as visões automaticamente. Consulte [estado da integração](../../../docs/estado_integracao.md) e [pgvector](../../../docs/legado/pgvector.md). Criar apenas o contêiner não restaura as coleções.


In [ ]:
# Altere estas opções e reexecute esta célula antes da execução.
EXECUTAR_MODELOS = False
EXECUTAR_TESTES = False
SALVAR_RESULTADO = True

MODO = "sintetico"  # "sintetico" ou "real"
CASO = "ferias"    # "ferias", "revisao_controlada", "sem_evidencia" ou "fora_escopo"
PERGUNTAS = {
    "ferias": "Como organizar a leitura de informações sobre férias, de acordo com o material fornecido?",
    "revisao_controlada": "Como organizar a leitura de informações sobre férias, de acordo com o material fornecido?",
    "sem_evidencia": "O que as fontes fornecidas dizem sobre férias?",
    "fora_escopo": "Como rescindir um contrato de aluguel residencial?",
}
PERGUNTA = PERGUNTAS[CASO]

URL_OLLAMA = "http://127.0.0.1:11434"
MODELO_ANALISTA = "qwen3:8b-q8_0"
MODELO_VALIDADOR = "gemma3:12b"
TIMEOUT_SEGUNDOS = 180
TEMPO_MAXIMO_SEGUNDOS = 900

estado = None
diretorio_resultado = None
print({"executar_modelos": EXECUTAR_MODELOS, "modo": MODO, "caso": CASO})
print("Pergunta:", PERGUNTA)


## O que demonstrar em cada caso

| Caso | Preparação | Comportamento a observar |
|---|---|---|
| `ferias` | Um material educativo explicitamente fictício sobre organizar a leitura | Consulta escolhida pelo Analista, rascunho apoiado no material e conferência pelo Validador |
| `revisao_controlada` | Mesmo material; o orquestrador altera uma citação do primeiro candidato para `[Fonte 99]` | Falha marcada como injetada, feedback de correção, revisão e nova validação |
| `sem_evidencia` | A busca devolve lista vazia | Encerramento com insuficiência de evidência, sem inventar regra jurídica |
| `fora_escopo` | Pergunta sobre aluguel | Encaminhamento pelo guardrail, sem chamadas desnecessárias aos agentes |

O material sintético não define prazos, valores, direitos ou entendimentos judiciais. Ele serve para demonstrar o fluxo e não mede qualidade jurídica. Para `ferias`, a pergunta deve tratar da organização da leitura desse material; pedir uma conclusão jurídica pode levar corretamente à abstenção. A revisão controlada testa a reação a uma falha inserida e não deve ser apresentada como erro natural do modelo.

Nos casos sem evidência e fora do escopo, encerrar cedo é o comportamento esperado; eles não demonstram, isoladamente, colaboração completa entre os dois agentes. Para isso, inspecione uma execução com rascunho, validação e, quando ocorrer, revisão.

O modo real usa `PERGUNTA` e as coleções existentes. A injeção de falha é permitida somente no modo sintético. Os casos sintéticos não garantem o mesmo resultado com modelos diferentes.


In [ ]:
# Conferência de presença; não acessa serviços nem lê credenciais.
print("Python:", sys.version.split()[0])
print("LangGraph instalado:", importlib.util.find_spec("langgraph") is not None)
print("Binário Ollama encontrado:", shutil.which("ollama") is not None)
if MODO == "real":
    for relativo in [
        ".env.pgvector.local",
        "dados/processed/legislacao/legislacao_artigos_completos.json",
        "dados/chunks/legislacao/chunks_D_paragrafo_completo.json",
    ]:
        print(relativo, "presente" if (ROOT / relativo).exists() else "ausente")
    print("A presença desses arquivos não comprova banco, modelos ou coleções disponíveis.")
else:
    print("A busca sintética dispensa banco e embeddings; a execução dos agentes exige Ollama e modelos reais.")


In [ ]:
if not EXECUTAR_MODELOS:
    print("Execução desativada. Prepare Ollama/modelos e altere EXECUTAR_MODELOS para True na configuração.")
else:
    # Importações e criação dos componentes reais somente dentro da execução habilitada.
    from worker.evidencia.langgraph.cliente_llm import ClienteOllama
    from worker.evidencia.langgraph.ferramentas import BuscaReal, BuscaSintetica
    from worker.evidencia.langgraph.orquestrador import ConfiguracaoSistema, SistemaMultiagentes
    from worker.evidencia.langgraph.registro import salvar_execucao

    if MODO not in ("sintetico", "real") or CASO not in PERGUNTAS:
        raise ValueError("Confira MODO e CASO na célula de configuração.")
    if MODO == "real" and CASO == "revisao_controlada":
        raise ValueError("O teste de falha controlada exige MODO = 'sintetico'.")

    config = ConfiguracaoSistema(
        modelo_analista=MODELO_ANALISTA,
        modelo_validador=MODELO_VALIDADOR,
        max_chamadas_llm=6,
        max_revisoes=1,
        max_tentativas_tecnicas=1,
        timeout_segundos=TIMEOUT_SEGUNDOS,
        tempo_maximo_segundos=TEMPO_MAXIMO_SEGUNDOS,
    )
    cliente = ClienteOllama(url=URL_OLLAMA, timeout=TIMEOUT_SEGUNDOS)
    busca = (
        BuscaSintetica(caso="sem_evidencia" if CASO == "sem_evidencia" else "ferias")
        if MODO == "sintetico" else BuscaReal(ROOT, device="cpu")
    )
    try:
        sistema = SistemaMultiagentes(cliente, busca, config=config)
        estado = sistema.executar(
            PERGUNTA, modo=MODO, caso=CASO if MODO == "sintetico" else None,
        )
    finally:
        busca.fechar()

    if SALVAR_RESULTADO:
        diretorio_resultado = salvar_execucao(estado, ROOT / "dados/output/multiagentes")
        print("Execução registrada em:", diretorio_resultado)
    print("Status:", estado["status"], "| motivo:", estado["motivo"])


## Inspecionar a execução

As células abaixo mostram somente o que foi produzido na execução habilitada. O trace descreve chamadas, ações, fontes e decisões operacionais; não solicita nem exibe cadeia de pensamento.

Aprovação significa passagem pelos controles do protótipo. `limitada` pode indicar insuficiência de evidência ou revisão esgotada; `falha_tecnica` exige verificar o motivo registrado e o ambiente. Nenhum desses estados deve ser convertido em resultado aprovado para a apresentação.


In [ ]:
if estado is None:
    print("Ainda não há execução. Habilite os modelos e execute a célula anterior para inspecionar resultados reais.")
else:
    campos = [
        "run_id", "modo", "caso", "status", "motivo", "chamadas_llm", "buscas_usadas",
        "revisoes_usadas", "tentativas_tecnicas", "tokens_entrada", "tokens_saida", "segundos", "falha_injetada",
    ]
    print(json.dumps({c: estado.get(c) for c in campos}, ensure_ascii=False, indent=2))
    print("\nEventos operacionais:")
    for evento in estado.get("eventos", []):
        print(json.dumps(evento, ensure_ascii=False))
    snapshot = estado.get("fontes_snapshot", {})
    print("\nHash do snapshot das fontes:", snapshot.get("sha256", "sem busca"))
    for numero, fonte in enumerate(snapshot.get("fontes", []), 1):
        print({"fonte": numero, "unidade": fonte.get("unidade"), "titulo": fonte.get("titulo"),
               "sintetica": fonte.get("sintetica", False)})


In [ ]:
if estado is None:
    print("Rascunhos, feedback e resposta final serão mostrados após uma execução real dos agentes.")
else:
    for versao in estado.get("historico", []):
        print("\n--- Versão", versao["versao"], "---")
        print("Rascunho do Analista:\n", versao.get("rascunho", ""))
        if versao.get("candidato") and versao["candidato"] != versao.get("rascunho"):
            print("Candidato após preparação/controles:\n", versao["candidato"])
        print("Guardrails:", json.dumps(versao.get("guardrails"), ensure_ascii=False))
        print("Feedback:", json.dumps(versao.get("feedback"), ensure_ascii=False, indent=2))
    print("\n--- Resposta final ---\n", estado["resposta_final"])
    if not estado.get("historico"):
        print("Não houve rascunho: confira o encerramento por escopo, evidência ou falha técnica no trace.")


## Evidências para a apresentação

Em uma execução com revisão, conferir a sequência **consulta do Analista → ferramenta → rascunho v0 → avaliação → feedback → revisão v1 → nova avaliação**. Mostrar o problema apontado e a mudança do texto, mantendo o mesmo snapshot de fontes. A resposta anterior permanece no histórico.

O limite é de uma busca, uma revisão, uma tentativa técnica adicional e seis chamadas de modelo no total, conforme a configuração. A tentativa técnica trata falha de execução/contrato; ela não é uma revisão semântica. O orquestrador não libera um candidato com regra bloqueante só porque o Validador pediu aprovação.

Arquivos persistidos pertencem à execução local. No modo real, podem conter fontes e textos com dados pessoais; devem ser mantidos fora do versionamento público. Para compartilhar o notebook, remova as saídas de uma execução real. Este arquivo inicial não contém evidências nem saídas fabricadas.


In [ ]:
if not EXECUTAR_TESTES:
    print("Testes desativados. Altere EXECUTAR_TESTES para True para rodar a suíte do protótipo.")
else:
    verificacao = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_multiagentes*.py"],
        cwd=ROOT, capture_output=True, text=True, check=False,
    )
    print(verificacao.stdout)
    print(verificacao.stderr)
    if verificacao.returncode:
        raise RuntimeError("A suíte do protótipo apresentou falhas; confira a saída acima.")


Os testes automatizados podem usar clientes simulados para verificar contratos, orçamento, segurança dos erros e transições. Eles não substituem as chamadas reais da demonstração nem comprovam qualidade jurídica. Os resultados anteriores de recuperação e guardrails do projeto também não medem, por si só, a qualidade deste ciclo de colaboração.

A execução com os dois modelos depende da memória e do tempo disponíveis. O cliente descarrega cada modelo após a chamada (`keep_alive=0`) para permitir uso sequencial; isso não garante que cada modelo caiba no equipamento. Falhas devem permanecer registradas como falhas, sem inserir respostas prontas para produzir uma aparência de sucesso.
